In [18]:
import requests
from bs4 import BeautifulSoup
import re
import sqlite3
import time
import json

WIKI_BASE = "https://en.wikipedia.org"
URL = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"
HEADERS = {"User-Agent": "Mozilla/5.0 (educational project)"}

In [3]:
def clean_money(raw: str) -> float | None:
    if not raw:
        return None

    match = re.search(r'\$[\d,]+', raw)
    if not match:
        return None

    num_str = match.group().replace('$', '').replace(',', '')

    try:
        return float(num_str)
    except ValueError:
        return None


def clean_year(raw: str) -> int | None:
    match = re.search(r'(\d{4})', raw)
    return int(match.group(1)) if match else None


def fetch_soup(url: str) -> BeautifulSoup:
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, 'html.parser')


def get_director_and_country(film_url: str) -> tuple[str, str]:
    director, country = None, None
    try:
        soup = fetch_soup(WIKI_BASE + film_url)
        infobox = soup.find("table", class_="infobox")
        if not infobox:
            return director, country

        rows = infobox.find_all("tr")
        for row in rows:
            header = row.find("th")
            if not header:
                continue
            label = header.get_text(strip=True).lower()

            if "directed" in label or label == "director":
                td = row.find("td")
                if td:
                    names = [a.get_text(strip=True) for a in td.find_all("a")]
                    if not names:
                        names = [td.get_text(strip=True)]
                    director = ", ".join(names)

            if label in ("country", "countries"):
                td = row.find("td")
                if td:
                    country = td.get_text(" ", strip=True)

    except Exception as e:
        print(f'Could not fetch details from {film_url}: {e}')

    return director, country

In [4]:
soup = fetch_soup(URL)

tables = soup.find_all("table", class_="wikitable")

main_table = None
for tbl in tables:
    first_row_headers = [th.get_text(strip=True) for th in tbl.find_all("th")]
    if "Rank" in first_row_headers and "Title" in first_row_headers:
        main_table = tbl
        break

if main_table is None:
    main_table = tables[0]
    print("Could not locate the expected table; using the first wikitable.")

print("Main table located.")

Main table located.


In [ ]:
films = []

rows = main_table.find_all("tr")[1:]

for row in rows:
    cells = row.find_all(["th", "td"])
    if len(cells) < 4:
        continue

    # --- Title & link -------------------------------------------------
    title_cell = cells[2]                       # 2nd column
    title_tag  = title_cell.find("a")
    title      = title_cell.get_text(strip=True)
    film_link  = title_tag["href"] if title_tag and title_tag.has_attr("href") else None

    # --- Worldwide gross ----------------------------------------------
    gross_raw  = cells[3].get_text(strip=True)
    box_office = clean_money(gross_raw)

    # --- Year ---------------------------------------------------------
    year_raw   = cells[4].get_text(strip=True)
    year       = clean_year(year_raw)

    films.append({
        "title":      title,
        "year":       year,
        "box_office": box_office,
        "link":       film_link,
        "director":   None,
        "country":    None,
    })

print(f"✔ Extracted {len(films)} films from the main table.")

print(films)

✔ Extracted 50 films from the main table.
[{'title': 'Avatar', 'year': 2009, 'box_office': 2923710708.0, 'link': '/wiki/Avatar_(2009_film)', 'director': None, 'country': None}, {'title': 'Avengers: Endgame', 'year': 2019, 'box_office': 2797501328.0, 'link': '/wiki/Avengers:_Endgame', 'director': None, 'country': None}, {'title': 'Avatar: The Way of Water', 'year': 2022, 'box_office': 2334484620.0, 'link': '/wiki/Avatar:_The_Way_of_Water', 'director': None, 'country': None}, {'title': 'Titanic', 'year': 1997, 'box_office': 2257906828.0, 'link': '/wiki/Titanic_(1997_film)', 'director': None, 'country': None}, {'title': 'Ne Zha 2', 'year': 2025, 'box_office': 2215690000.0, 'link': '/wiki/Ne_Zha_2', 'director': None, 'country': None}, {'title': 'Star Wars: The Force Awakens', 'year': 2015, 'box_office': 2068223624.0, 'link': '/wiki/Star_Wars:_The_Force_Awakens', 'director': None, 'country': None}, {'title': 'Avengers: Infinity War', 'year': 2018, 'box_office': 2048359754.0, 'link': '/wiki/

In [9]:
for i, film in enumerate(films):
    if film["link"]:
        director, country = get_director_and_country(film["link"])
        film["director"] = director
        film["country"]  = country
        print(f"  [{i+1}/{len(films)}] {film['title']:40s} -> dir: {director}")
        time.sleep(0.5)                         # rate-limit

print("Director/country enrichment complete.")

  [1/50] Avatar                                   -> dir: James Cameron
  [2/50] Avengers: Endgame                        -> dir: Anthony RussoJoe Russo
  [3/50] Avatar: The Way of Water                 -> dir: James Cameron
  [4/50] Titanic                                  -> dir: James Cameron
  [5/50] Ne Zha 2                                 -> dir: Jiaozi
  [6/50] Star Wars: The Force Awakens             -> dir: J. J. Abrams
  [7/50] Avengers: Infinity War                   -> dir: Anthony RussoJoe Russo
  [8/50] Spider-Man: No Way Home                  -> dir: Jon Watts
  [9/50] Zootopia 2†                              -> dir: Jared Bush, Byron Howard
  [10/50] Inside Out 2                             -> dir: Kelsey Mann
  [11/50] Jurassic World                           -> dir: Colin Trevorrow
  [12/50] The Lion King                            -> dir: Jon Favreau
  [13/50] The Avengers                             -> dir: Joss Whedon
  [14/50] Furious 7                            

In [10]:
DB_NAME = "highest_grossing_films.db"

conn = sqlite3.connect(DB_NAME)
cur  = conn.cursor()

cur.execute("DROP TABLE IF EXISTS films;")
cur.execute("""
    CREATE TABLE films (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        title        TEXT    NOT NULL,
        release_year INTEGER,
        director     TEXT,
        box_office   REAL,
        country      TEXT
    );
""")

insert_sql = """
    INSERT INTO films (title, release_year, director, box_office, country)
    VALUES (?, ?, ?, ?, ?);
"""
for f in films:
    cur.execute(insert_sql, (
        f["title"],
        f["year"],
        f["director"],
        f["box_office"],
        f["country"],
    ))

conn.commit()
print(f"Inserted {len(films)} rows into '{DB_NAME}'.")

Inserted 50 rows into 'highest_grossing_films.db'.


In [17]:
import pandas as pd


conn =  sqlite3.connect(DB_NAME)
df = pd.read_sql_query("SELECT * FROM films ORDER BY box_office DESC LIMIT 10;", conn)
print("── Top 10 by Box Office ──")
display(df)

# 7b. Count of films per decade
df_decade = pd.read_sql_query("""
    SELECT (release_year / 10) * 10 AS decade,
           COUNT(*)                  AS film_count,
           PRINTF('$%,.0f', AVG(box_office)) AS avg_gross
    FROM   films
    GROUP  BY decade
    ORDER  BY decade;
""", conn)
print("\n── Films per Decade ──")
display(df_decade)

# 7c. Films directed by a specific director (example)
df_dir = pd.read_sql_query("""
    SELECT title, release_year, box_office
    FROM   films
    WHERE  director LIKE '%Cameron%'
    ORDER  BY box_office DESC;
""", conn)
print("\n── Spielberg Films ──")
display(df_dir)

conn.close()
print("Database connection closed.")

── Top 10 by Box Office ──


,id,title,release_year,director,box_office,country
0,1,Avatar,2009,James Cameron,2.923711e+09,United States [ 1 ] [ 3 ] United Kingdom [ 1 ]...
1,2,Avengers: Endgame,2019,Anthony RussoJoe Russo,2.797501e+09,United States
2,3,Avatar: The Way of Water,2022,James Cameron,2.334485e+09,United States
3,4,Titanic,1997,James Cameron,2.257907e+09,United States
4,5,Ne Zha 2,2025,Jiaozi,2.215690e+09,China
5,6,Star Wars: The Force Awakens,2015,J. J. Abrams,2.068224e+09,United States
6,7,Avengers: Infinity War,2018,Anthony RussoJoe Russo,2.048360e+09,United States
7,8,Spider-Man: No Way Home,2021,Jon Watts,1.922599e+09,United States
8,9,Zootopia 2†,2025,"Jared Bush, Byron Howard",1.866578e+09,United States
9,10,Inside Out 2,2024,Kelsey Mann,1.698864e+09,United States



── Films per Decade ──


,decade,film_count,avg_gross
0,1990,2,$1652211119
1,2000,3,$1712629287
2,2010,34,$1332134449
3,2020,11,$1656804640



── Spielberg Films ──


,title,release_year,box_office
0,Avatar,2009,2.923711e+09
1,Avatar: The Way of Water,2022,2.334485e+09
2,Titanic,1997,2.257907e+09
3,Avatar: Fire and Ash†,2025,1.485605e+09


Database connection closed.


In [ ]:
JSON_NAME = "data/films.json"

conn = sqlite3.connect(DB_NAME)
conn.row_factory = sqlite3.Row          # rows behave like dicts
cur = conn.cursor()

cur.execute("""
    SELECT id,
           title,
           release_year,
           director,
           box_office,
           country
    FROM   films
    ORDER  BY box_office DESC;
""")

films = [dict(row) for row in cur.fetchall()]
conn.close()

# Make sure the data/ folder exists
import os
os.makedirs("data", exist_ok=True)

with open(JSON_NAME, "w", encoding="utf-8") as f:
    json.dump(films, f, indent=2, ensure_ascii=False)

print(f"Exported {len(films)} films to {JSON_NAME}")

✔ Exported 50 films to data/films.json
